<a href="https://colab.research.google.com/github/enjomcoding/Earthquake_Decision-Tree/blob/main/Tijada_Decision_Tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Decision Tree Modeling for Post-Earthquake Structural Damage**
This project utilizes a decision tree classifier to assess seismic vulnerability in buildings. By analyzing a comprehensive set of 38 tabular features, ranging from geographic location IDs and physical dimensions (such as age, area, and number of floors) to specific construction materials (like foundation type, roof type, and mud-mortar stone superstructures), the model predicts the specific damage grade a structure is likely to sustain during an earthquake. This provides a clear, data-driven approach to evaluating post-earthquake structural health and risk.

# **1. Preparing the Environment**

**1.1 Install Pycaret**

In [ ]:
!pip install git+https://github.com/pycaret/pycaret.git@master --upgrade

**1.2 Import required libraries and modules**

In [ ]:
import sys
import pandas as pd
import pycaret
from pycaret.classification import *
import pycaret.utils.generic
import pycaret.internal.metrics

**1.3 Load dataset**

In [ ]:
# Load the datasets
features = pd.read_csv('train_values.csv')
labels = pd.read_csv('train_labels.csv')

# Merge into a single dataframe
seis_data = pd.merge(features, labels, on='building_id')

**1.5 Display first five observations**

In [ ]:
seis_data.head()

# **2. Understanding the Data**

**2.1 Display the number of observations, column names, and data types**

In [ ]:
seis_data.info()

**2.2 Display the summary statistics**

In [ ]:
seis_data.describe()

In [ ]:
seis_data.describe(exclude='number')

**2.3 Display the distribution plots of the variables**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 5))

# Plot for the target variable
plt.subplot(1, 2, 1)
sns.countplot(data=seis_data, x='damage_grade')
plt.title('Distribution of Damage Grade')

# Plot for a feature
plt.subplot(1, 2, 2)
sns.countplot(data=seis_data, x='foundation_type')
plt.title('Distribution of Foundation Types')

plt.tight_layout()
plt.show()

# **3. Data preprocessing**

**3.1 Remove Outliers**

**3.2 Apply normalization**

**3.3 Impute missing values**

**3.4 Apply transformation**

**3.5 Split the dataset into train and test data using 80:20 ratio**

In [ ]:
pycaret.utils = sys.modules['pycaret.utils']
pycaret.internal = sys.modules['pycaret.internal']

clf_setup = setup(data = seis_data,
                  target = 'damage_grade',
                  train_size = 0.8,
                  session_id = 7402,
                  remove_outliers = True,
                  normalize = True,
                  normalize_method = 'minmax',
                  numeric_imputation = 'mean',
                  categorical_imputation = 'mode',
                  transformation = True)

**3.5 Display the preprocessed data**

In [ ]:
get_config('dataset_transformed').head()

# **4. Modelling**

**4.1 Compare different classification models**

In [ ]:
classification = compare_models(sort='Accuracy')

**4.2 Create a Decision Tree model**

In [ ]:
DT = create_model('dt')

# **5.0 Evaluate the model**

**5.1 Make predictions using the Decision Tree model**

In [ ]:
dt_predictions = predict_model(DT)

**5.2 Display prediction scores**

In [ ]:
dt_predictions.head()

**5.3 Plot the confusion matrix**

In [ ]:
plot_model(DT, plot = 'confusion_matrix')

**5.4 Plot the model**

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 10))

plot_tree(DT,
          filled=True,
          rounded=True,
          class_names=['Grade 1', 'Grade 2', 'Grade 3'],
          max_depth=3)

plt.title("Decision Tree for Earthquake Damage Classification")
plt.show()

**5.5 Finalize and save the model**

In [ ]:
final_model = finalize_model(DT)
save_model(final_model, 'earthquake_dt_model')

# **Sample Decision**

In [ ]:
sample_building = features.iloc[[0]]

print("Here is the building's raw data:")
display(sample_building)

prediction = predict_model(DT, data=sample_building)

print("\nHere is the model's prediction:")
display(prediction)

print("Building Analysis Results:")
print(f"Assigned Damage Grade: {prediction['prediction_label'].values[0]}")
print(f"Model Confidence: {prediction['prediction_score'].values[0] * 100:.2f}%")

# Simple logic to print the status
grade = prediction['prediction_label'].values[0]
if grade == 1:
    print("Status: Safe / Low Risk")
elif grade == 2:
    print("Status: Restricted Access / Medium Risk")
else:
    print("Status: Unsafe / High Risk")